<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur les Champs Élysées.

Imports

In [24]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'champs_elysees.csv'

df_final = pd.read_csv('../datasets_axes_with_all_features/'+ doc, sep=';')

In [25]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8723 entries, 0 to 8722
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 8723 non-null   int64  
 1   Identifiant arc            8723 non-null   int64  
 2   Libelle                    8723 non-null   object 
 3   Date et heure de comptage  8723 non-null   object 
 4   Débit horaire              8174 non-null   float64
 5   Taux d'occupation          8159 non-null   float64
 6   Etat trafic                8723 non-null   object 
 7   Identifiant noeud amont    8723 non-null   int64  
 8   Libelle noeud amont        8723 non-null   object 
 9   Identifiant noeud aval     8723 non-null   int64  
 10  Libelle noeud aval         8723 non-null   object 
 11  Etat arc                   8723 non-null   object 
 12  Date debut dispo data      8723 non-null   object 
 13  Date fin dispo data        8723 non-null   objec

In [34]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'est_vacances', 'duree prec (en min)',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'jour_semaine', 'mois',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'est_vacances',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'duree prec (en min)', 'mois',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=4,
    max_iter=400,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
POST_SCALE_FERIE = 1.0

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
mask_est_pieton_test = X_test['est_pieton'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE
#y_pred[mask_est_pieton_test] *= 0.5

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

df_final['Débit_prédit'] = np.nan
df_final.loc[X_test.index, 'Débit_prédit'] = y_pred


R² : 0.784
MAE : 83.61
RMSE : 117.91
Part d'observations piéton (test) : 2.0%
MAE (jours piéton) : 185.59 (n=32)
MAE (jours non piéton) : 81.57 (n=1603)


In [35]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8723 entries, 0 to 8722
Data columns (total 43 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Unnamed: 0                 8723 non-null   int64         
 1   Identifiant arc            8723 non-null   int64         
 2   Libelle                    8723 non-null   object        
 3   Date et heure de comptage  8723 non-null   datetime64[ns]
 4   Débit horaire              8174 non-null   float64       
 5   Taux d'occupation          8159 non-null   float64       
 6   Etat trafic                8723 non-null   object        
 7   Identifiant noeud amont    8723 non-null   int64         
 8   Libelle noeud amont        8723 non-null   object        
 9   Identifiant noeud aval     8723 non-null   int64         
 10  Libelle noeud aval         8723 non-null   object        
 11  Etat arc                   8723 non-null   object        
 12  Date d

In [36]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
3                  heure_sin       170.624569        4.514929
4                  heure_cos       127.134466        3.198026
5                   jour_sin        77.048052        2.646698
15                est_pieton        14.098950        1.560899
9   force moyenne vent (m/s)         3.062039        0.488732
6                   jour_cos         2.604847        0.377239
12                 est_ferie         1.806154        1.461814
10              jour_semaine         1.502673        0.379846
14   ensoleillement (en min)         0.779455        0.227286
0                Température         0.226249        0.326387
13           est_avant_ferie         0.039858        0.058142
11                      mois        -0.173631        0.048538
2        duree prec (en min)        -0.320628        0.202712
7                   mois_sin        -1.373827        0.137680
1               est_vacances        -1.573443        0.524272
8       

In [37]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

<h2>Taux d'occupation</h2>

In [90]:
target_occ = 'Taux d\'occupation'

features_occ = [
    'Température', 'duree prec (en min)', 'Débit_prédit',
    'heure_sin', 'heure_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'mois', 'jour_sin',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]

mask_occ = df_final[target_occ].notna() & df_final['Débit_prédit'].notna()
X_occ = df_final.loc[mask_occ, features_occ].copy()
y_occ = df_final.loc[mask_occ, target_occ].astype(float)

X_train_occ, X_test_occ, y_train_occ, y_test_occ = train_test_split(
    X_occ, y_occ, test_size=0.2, shuffle=False
)

numeric_features_occ = [c for c in features_occ if c != 'jour_semaine']
categorical_features_occ = []

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess_occ = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features_occ),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features_occ),
    ],
    remainder='drop'
)

model_occ = HistGradientBoostingRegressor(
    loss='poisson',
    max_depth=5,
    max_iter=40,
    early_stopping=False,
    random_state=42
)

pipe_occ = Pipeline(steps=[('prep', preprocess_occ), ('model', model_occ)])

pipe_occ.fit(X_train_occ, y_train_occ)

y_pred_occ = pipe_occ.predict(X_test_occ)

print("=== Performances taux d'occupation ===")
print(f"R²   : {r2_score(y_test_occ, y_pred_occ):.3f}")
print(f"MAE  : {mean_absolute_error(y_test_occ, y_pred_occ):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_occ, y_pred_occ)):.2f}")
print(f"nb test : {len(y_test_occ)}")


=== Performances taux d'occupation ===
R²   : 0.826
MAE  : 2.66
RMSE : 3.50
nb test : 327


In [91]:
perm = permutation_importance(
    estimator=pipe_occ,
    X=X_test_occ,
    y=y_test_occ,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  
)

imp_df = (
    pd.DataFrame({
        'feature': features_occ,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
2               Débit_prédit         3.624074        0.132023
3                  heure_sin         2.259422        0.142546
4                  heure_cos         0.702066        0.113012
7   force moyenne vent (m/s)         0.324264        0.109128
5                   mois_sin         0.282888        0.069288
9                   jour_sin         0.267391        0.054516
12   ensoleillement (en min)         0.102744        0.026711
0                Température         0.047505        0.030518
6                   mois_cos         0.000000        0.000000
8                       mois         0.000000        0.000000
10                 est_ferie         0.000000        0.000000
11           est_avant_ferie         0.000000        0.000000
13                est_pieton         0.000000        0.000000
1        duree prec (en min)        -0.003579        0.004949


In [92]:
time_index_occ = df_final.loc[X_test_occ.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_pred_occ,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_test_occ,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()